# Task 2: paired class-weight check

Question: does giving Social (2) and Whale (3) more training weight improve macro F1 across the *same five development folds*? Original 41 non-noise predictors; unchanged CatBoost settings otherwise. The prior 20% holdout is reported separately and has already been reused in earlier rounds. No Test fitting or tuning.

In [1]:
from pathlib import Path
from hashlib import sha256
import json, sys, joblib
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src'/'pipeline'/'save_data.py').exists())
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
SRC=ROOT/'src'
import numpy as np, pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix, precision_recall_fscore_support
from catboost import CatBoostClassifier
from src.pipeline.random_seed import set_seed
from src.pipeline.save_model import save_model
from src.pipeline.evaluate_model import evaluate_model
from src.task2.models.decision_rules import restore_categorical_nulls
SEED=set_seed()
DATA_NAME='eda_original_v2'; BASE='t2_cat_original_no_noise_v2'
data_card=json.loads((SRC/'task2'/'data'/'eda'/f'{DATA_NAME}.json').read_text(encoding='utf8'))
base_card=json.loads((SRC/'task2'/'save'/f'{BASE}.json').read_text(encoding='utf8'))
assert data_card['task']=='task2' and base_card['data_name']==DATA_NAME
path=SRC/data_card['splits']['train']['path']
train=pd.read_parquet(path) if data_card['format']=='parquet' else pd.read_csv(path)
assert train.player_id.is_unique and train.segment.notna().all()
y=train.segment.astype(int)
X0=train.drop(columns=['player_id','segment'])
categorical=X0.select_dtypes(include=['object','str']).columns.tolist()
assert len(X0.columns)==44 and len(categorical)==10
X=restore_categorical_nulls(X0,categorical)
dev_idx,valid_idx=train_test_split(np.arange(len(y)),test_size=.20,stratify=y,random_state=SEED)
X_dev,X_valid=X.iloc[dev_idx],X.iloc[valid_idx]
y_dev,y_valid=y.iloc[dev_idx],y.iloc[valid_idx]
cv_X,cv_y=X_dev.reset_index(drop=True),y_dev.reset_index(drop=True)
splits=list(StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED).split(cv_X,cv_y))
signature=sha256(np.asarray(dev_idx,dtype=np.int64).tobytes()+np.asarray(y_dev,dtype=np.int8).tobytes()).hexdigest()
baseline=joblib.load(SRC/base_card['model_path'])
base_params=baseline.named_steps['classifier'].get_params()
assert base_params.get('class_weights') is None and base_params['random_seed']==SEED
assert X.columns.tolist()==baseline.named_steps['preprocess'].feature_names_in_.tolist()
print('Development:',len(cv_X),'holdout:',len(X_valid),'features:',len(X.columns))

Development: 83200 holdout: 20800 features: 44


## Existing out-of-fold errors

Use the existing fold predictions solely as a baseline. These are development rows, each predicted by a model fitted without that fold.

In [2]:
RESULTS=SRC/'task2'/'models'/'results'
cache_key={'data_name':DATA_NAME,'data_created_at':data_card['created_at'],
    'model_name':BASE,'model_created_at':base_card['created_at'],'seed':SEED,
    'folds':5,'split_signature':signature}
baseline_parts=[]
for fold,(_,check_pos) in enumerate(splits,start=1):
    p=RESULTS/f'class_bias_fold{fold}_v1.parquet'
    m=RESULTS/f'class_bias_fold{fold}_v1.json'
    assert p.exists() and m.exists() and json.loads(m.read_text(encoding='utf8'))=={'key':cache_key,'fold':fold}
    part=pd.read_parquet(p)
    assert part.row_pos.tolist()==check_pos.tolist()
    baseline_parts.append(part)
base_oof=pd.concat(baseline_parts,ignore_index=True)
assert base_oof.row_pos.is_unique and len(base_oof)==len(cv_X)
base_oof['pred']=base_oof[[f'p_{k}' for k in range(4)]].to_numpy().argmax(axis=1)
display(pd.DataFrame(confusion_matrix(base_oof.true,base_oof.pred,labels=[0,1,2,3]),
    index=['true Casual','true Grinder','true Social','true Whale'],
    columns=['pred Casual','pred Grinder','pred Social','pred Whale']))
print(classification_report(base_oof.true,base_oof.pred,digits=4))
print('High confidence mistakes (max P > 0.8):',
      int(((base_oof[[f'p_{k}' for k in range(4)]].max(axis=1)>.8)&
           (base_oof.true!=base_oof.pred)).sum()))

,pred Casual,pred Grinder,pred Social,pred Whale
true Casual,28265,2779,1525,197
true Grinder,4098,15275,570,882
true Social,3155,831,11410,1427
true Whale,531,1507,1813,8935


              precision    recall  f1-score   support

           0     0.7841    0.8626    0.8215     32766
           1     0.7491    0.7335    0.7412     20825
           2     0.7449    0.6782    0.7100     16823
           3     0.7810    0.6988    0.7376     12786

    accuracy                         0.7678     83200
   macro avg     0.7647    0.7433    0.7526     83200
weighted avg     0.7669    0.7678    0.7660     83200

High confidence mistakes (max P > 0.8): 2722


## Weight Social and Whale

A single prespecified weight of 1.25 for each undercalled class. Every fold re-fits preprocessing and CatBoost on its own training rows. Fold predictions are cached with dataset/model/split/weight lineage to allow review without retraining.

In [3]:
WEIGHTS=[1.0,1.0,1.25,1.25]
NAME='t2_cat_weighted_23_cv5_v4'
weighted_key={**cache_key,'experiment':NAME,'weights':WEIGHTS}
weighted_parts=[]
for fold,(fit_pos,check_pos) in enumerate(splits,start=1):
    p=RESULTS/f'weighted_23_fold{fold}_v4.parquet'
    m=RESULTS/f'weighted_23_fold{fold}_v4.json'
    if p.exists() and m.exists():
        assert json.loads(m.read_text(encoding='utf8'))=={'key':weighted_key,'fold':fold}
        part=pd.read_parquet(p)
        assert part.row_pos.tolist()==check_pos.tolist()
        print('Loaded fold',fold)
    else:
        assert not (p.exists() or m.exists()),'Inspect incomplete cache'
        params={**base_params,'class_weights':WEIGHTS}
        model=Pipeline([('preprocess',clone(baseline.named_steps['preprocess'])),
                        ('classifier',CatBoostClassifier(**params))])
        model.fit(cv_X.iloc[fit_pos],cv_y.iloc[fit_pos])
        probabilities=np.asarray(model.predict_proba(cv_X.iloc[check_pos]),dtype=float)
        part=pd.DataFrame({'fold':fold,'row_pos':check_pos,'true':cv_y.iloc[check_pos].to_numpy(),
            **{f'p_{k}':probabilities[:,k] for k in range(4)}})
        part.to_parquet(p,index=False)
        m.write_text(json.dumps({'key':weighted_key,'fold':fold},indent=2)+'\n',encoding='utf8')
        print('Trained fold',fold)
    weighted_parts.append(part)
weighted_oof=pd.concat(weighted_parts,ignore_index=True)
assert weighted_oof.row_pos.is_unique and len(weighted_oof)==len(cv_X)
weighted_oof['pred']=weighted_oof[[f'p_{k}' for k in range(4)]].to_numpy().argmax(axis=1)
assert weighted_oof.row_pos.tolist()==base_oof.row_pos.tolist()
rows=[]
for fold in range(1,6):
    b=base_oof[base_oof.fold==fold];w=weighted_oof[weighted_oof.fold==fold]
    assert b.true.tolist()==w.true.tolist()
    old=f1_score(b.true,b.pred,average='macro')
    new=f1_score(w.true,w.pred,average='macro')
    rows.append({'fold':fold,'baseline_macro_F1':old,'weighted_macro_F1':new,'delta':new-old})
paired=pd.DataFrame(rows)
display(paired)
print('Mean paired baseline:',paired.baseline_macro_F1.mean())
print('Mean paired weighted:',paired.weighted_macro_F1.mean())
print('Mean gain:',paired.delta.mean(),'wins:',int((paired.delta>0).sum()),'/5')
print('Pooled OOF per-class F1 baseline:',
      f1_score(base_oof.true,base_oof.pred,average=None,labels=[0,1,2,3]).round(4))
print('Pooled OOF per-class F1 weighted:',
      f1_score(weighted_oof.true,weighted_oof.pred,average=None,labels=[0,1,2,3]).round(4))
display(pd.DataFrame(confusion_matrix(weighted_oof.true,weighted_oof.pred,labels=[0,1,2,3]),
    index=['true Casual','true Grinder','true Social','true Whale'],
    columns=['pred Casual','pred Grinder','pred Social','pred Whale']))
print(classification_report(weighted_oof.true,weighted_oof.pred,digits=4))

Trained fold 1


Trained fold 2


Trained fold 3


Trained fold 4


Trained fold 5


,fold,baseline_macro_F1,weighted_macro_F1,delta
0,1,0.756123,0.755899,-0.000224
1,2,0.746062,0.747641,0.001580
2,3,0.750865,0.753011,0.002147
3,4,0.758046,0.756484,-0.001562
4,5,0.751726,0.752465,0.000740


Mean paired baseline: 0.7525642419662718
Mean paired weighted: 0.7531000717248977
Mean gain: 0.0005358297586258942 wins: 3 /5
Pooled OOF per-class F1 baseline: [0.8215 0.7412 0.71   0.7376]
Pooled OOF per-class F1 weighted: [0.8201 0.7397 0.7121 0.7405]


,pred Casual,pred Grinder,pred Social,pred Whale
true Casual,27777,2759,2017,213
true Grinder,3982,15004,757,1082
true Social,2728,727,11884,1484
true Whale,486,1254,1895,9151


              precision    recall  f1-score   support

           0     0.7942    0.8477    0.8201     32766
           1     0.7599    0.7205    0.7397     20825
           2     0.7179    0.7064    0.7121     16823
           3     0.7671    0.7157    0.7405     12786

    accuracy                         0.7670     83200
   macro avg     0.7598    0.7476    0.7531     83200
weighted avg     0.7660    0.7670    0.7659     83200



## Saved development-fit model and reused holdout

This model uses development training only, matching the saved baseline's fit size. The holdout is an additional check, not a fresh selection set. A full-train refit would be required for a final submission.

In [4]:
SAVE=SRC/'task2'/'save'
model_path,card_path,eval_path=(SAVE/f'{NAME}.joblib',SAVE/f'{NAME}.json',
                                SAVE/f'{NAME}-evaluation.json')
if not any(p.exists() for p in (model_path,card_path,eval_path)):
    params={**base_params,'class_weights':WEIGHTS}
    weighted_model=Pipeline([('preprocess',clone(baseline.named_steps['preprocess'])),
                             ('classifier',CatBoostClassifier(**params))])
    weighted_model.fit(X_dev,y_dev)
    pred=np.asarray(weighted_model.predict(X_valid)).reshape(-1).astype(int)
    save_model(task=2,name=NAME,model=weighted_model,data_name=DATA_NAME,
        card={'idea':'Class weights for undercalled Social and Whale',
              'weights':WEIGHTS,'base_model':BASE,'seed':SEED,
              'cv':'5 paired development folds','mean_cv_macro_F1':float(paired.weighted_macro_F1.mean()),
              'mean_cv_gain':float(paired.delta.mean())})
    result=evaluate_model(task=2,name=NAME,y_true=y_valid,y_pred=pred,
        card={'split':'reused stratified 20% holdout','seed':SEED,
              'selection':'fixed weights checked on development folds'})
else:
    assert all(p.exists() for p in (model_path,card_path,eval_path)),'Inspect incomplete model artifact'
    saved=json.loads(card_path.read_text(encoding='utf8'))
    assert saved['data_name']==DATA_NAME
    weighted_model=joblib.load(model_path)
    pred=np.asarray(weighted_model.predict(X_valid)).reshape(-1).astype(int)
    result=json.loads(eval_path.read_text(encoding='utf8'))
base_pred=np.asarray(baseline.predict(X_valid)).reshape(-1).astype(int)
print('Baseline holdout macro F1:',f1_score(y_valid,base_pred,average='macro'))
print('Weighted holdout macro F1:',result['score'])
print('Changed holdout predictions:',int((base_pred!=pred).sum()))
print(classification_report(y_valid,pred,digits=4))

Baseline holdout macro F1: 0.7504144211991189
Weighted holdout macro F1: 0.7528493224569013
Changed holdout predictions: 659
              precision    recall  f1-score   support

           0     0.7909    0.8465    0.8178      8191
           1     0.7597    0.7323    0.7457      5207
           2     0.7222    0.6997    0.7108      4206
           3     0.7693    0.7074    0.7371      3196

    accuracy                         0.7669     20800
   macro avg     0.7605    0.7465    0.7528     20800
weighted avg     0.7659    0.7669    0.7657     20800



## Interpretation

Prefer the paired development folds for the weight decision. The pooled OOF and reused holdout reports explain which classes gained or lost. Stop this direction if gains are small or inconsistent; keep the baseline for submission unless evidence is strong.